# Mini-Project 2 — Part 1: Dataset Setup & Exploration
**Pascal VOC 2007 Semantic Segmentation**

## 1. Environment Setup

In [2]:
# Install dependencies (run once)
!pip install torch torchvision matplotlib numpy pillow scipy scikit-learn torchmetrics -q

## 2. Mount Google Drive & Extract Dataset

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

DRIVE_DIR = '/content/drive/MyDrive/SHBT261_mini_proj2'
DRIVE_ZIP = os.path.join(DRIVE_DIR, 'VOCtrainval_06-Nov-2007.zip')
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists(DRIVE_ZIP):
    print('Downloading Pascal VOC 2007 from Kaggle to Google Drive...')
    !pip install kaggle -q
    os.makedirs('/root/.kaggle', exist_ok=True)
    # Upload your kaggle.json via: from google.colab import files; files.upload()
    # Then move it: !mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
    !kaggle datasets download -d zaraks/pascal-voc-2007 -p {DRIVE_DIR}
    print(f'Dataset saved to Drive: {DRIVE_ZIP}')
else:
    print(f'Dataset zip already on Drive: {DRIVE_ZIP}')

Dataset URL: https://www.kaggle.com/datasets/zaraks/pascal-voc-2007
License(s): other
 55% 929M/1.65G [01:07<00:48, 16.3MB/s] 

In [ ]:
import shutil

# Copy zip from Drive to fast local SSD, then extract
LOCAL_DATA_DIR = '/content/VOCdata'

if not os.path.exists(LOCAL_DATA_DIR):
    print('Copying zip from Drive to local SSD...')
    shutil.copy(DRIVE_ZIP, '/content/VOCtrainval_06-Nov-2007.zip')
    print('Extracting...')
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    !unzip -q /content/VOCtrainval_06-Nov-2007.zip -d {LOCAL_DATA_DIR}
    print('Done.')
else:
    print('Dataset already extracted to local storage.')

# Root path used by VOCSegmentation loader
VOC_ROOT = LOCAL_DATA_DIR
print(f'VOC_ROOT = {VOC_ROOT}')

## 3. Define Classes & Transforms

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.datasets import VOCSegmentation
from torch.utils.data import DataLoader

VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus',
    'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike',
    'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]
NUM_CLASSES = len(VOC_CLASSES)
print(f'Number of classes: {NUM_CLASSES}')

IMG_SIZE = (256, 256)

transform_img = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

transform_target = transforms.Compose([
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
])

## 4. Load Datasets

In [ ]:
train_dataset = VOCSegmentation(
    root=VOC_ROOT, year='2007', image_set='train',
    download=False, transform=transform_img, target_transform=transform_target
)
# Per assignment: use val as test set
val_dataset = VOCSegmentation(
    root=VOC_ROOT, year='2007', image_set='val',
    download=False, transform=transform_img, target_transform=transform_target
)

print(f'Train samples : {len(train_dataset)}')
print(f'Test  samples : {len(val_dataset)}')

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

## 5. Dataset Exploration & Visualization

In [ ]:
# VOC colour palette (21 classes)
VOC_COLORMAP = np.array([
    [0,0,0],[128,0,0],[0,128,0],[128,128,0],[0,0,128],
    [128,0,128],[0,128,128],[128,128,128],[64,0,0],[192,0,0],
    [64,128,0],[192,128,0],[64,0,128],[192,0,128],[64,128,128],
    [192,128,128],[0,64,0],[128,64,0],[0,192,0],[128,192,0],[0,64,128]
], dtype=np.uint8)

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def denormalize(tensor):
    img = tensor.permute(1, 2, 0).numpy()
    return np.clip(img * STD + MEAN, 0, 1)

def mask_to_color(mask):
    mask = mask.squeeze().numpy().astype(np.uint8)
    mask = np.clip(mask, 0, 20)  # ignore=255 -> clip
    return VOC_COLORMAP[mask]

def show_samples(dataset, n=4):
    indices = np.random.choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
    for col, idx in enumerate(indices):
        img, mask = dataset[idx]
        axes[0, col].imshow(denormalize(img))
        axes[0, col].set_title(f'Image [{idx}]')
        axes[0, col].axis('off')
        axes[1, col].imshow(mask_to_color(mask))
        axes[1, col].set_title('Mask')
        axes[1, col].axis('off')
    plt.tight_layout()
    plt.show()

show_samples(train_dataset)

In [ ]:
# Class distribution in training set
from collections import Counter

class_counts = Counter()
for _, mask in train_dataset:
    vals = mask.unique().numpy().tolist()
    for v in vals:
        if v < NUM_CLASSES:
            class_counts[int(v)] += 1

labels = [VOC_CLASSES[i] for i in range(NUM_CLASSES)]
counts = [class_counts.get(i, 0) for i in range(NUM_CLASSES)]

plt.figure(figsize=(14, 4))
plt.bar(labels, counts, color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title('Class Frequency in Training Set (# images containing class)')
plt.tight_layout()
plt.show()

In [ ]:
# Save paths for reuse in other notebooks
import json
config = {
    'voc_root': VOC_ROOT,
    'img_size': list(IMG_SIZE),
    'num_classes': NUM_CLASSES,
    'voc_classes': VOC_CLASSES,
}
with open('/content/project_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Config saved to /content/project_config.json')